# 01 — Veri Hazırlama (İP-2)

Amaç: yönetmelik korpusundan chunk listesi üretmek ve soru–kanıt eşlemelerinin yazılacağı şemayı hazırlamak.

Önce `00_kurulum_ve_kontrol.ipynb` çalışmış olmalı (repo `/content/RAG-proje` altında).

In [1]:
import os, sys
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if os.path.exists('/content'):
    if not os.path.exists(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
    sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data import korpus_yukle, yonetmelik_parcala, chunklari_kaydet

cfg = load_config()
metin = korpus_yukle(cfg['veri']['korpus'])
chunklar = yonetmelik_parcala(
    metin,
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
chunklari_kaydet(chunklar, 'data/processed/chunklar.jsonl')
print(f'{len(chunklar)} chunk yazıldı: data/processed/chunklar.jsonl')
for c in chunklar[:3]:
    print(c.chunk_id, '|', c.metin[:90], '...')

Cloning into '/content/RAG-proje'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 89 (delta 25), reused 79 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 90.81 KiB | 4.78 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/RAG-proje
From https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri
 * branch            RAG-dev    -> FETCH_HEAD
Already up to date.
82 chunk yazıldı: data/processed/chunklar.jsonl
MADDE-1 | MADDE 1 - (1) Bu Yönetmeliğin amacı; Doğuş Üniversitesine bağlı fakülte, yüksekokul, mesle ...
MADDE-2 | MADDE 2 - (1) Bu Yönetmelik; Doğuş Üniversitesinin fakülte, yüksekokul, meslek yüksekokulu ...
MADDE-3 | MADDE 3 - (1) Bu Yönetmelik; 4/11/1981 tarihli ve 2547 sayılı Yükseköğretim Kanununun 14 ü ...


## Soru yazma görevi (Araştırmacı 1)

Her madde için 2–4 doğal soru yazın. Şema: `data/README.md`. Kritik kurallar:

- Her sorunun `kanit_chunk_id` alanı, cevabı içeren chunk'ı göstermeli.
- Aynı sorunun paraphrase varyantları aynı `varyant_grup` kimliğini alır ve **aynı bölmede** kalır.
- Soru tipleri: `olgusal`, `paraphrase`, `bicimbilimsel`, `cok-maddeli`.
- Hedef: 300–500 özgün soru. Örnek kayıtlar: `data/processed/sorular-ornek.jsonl`.

In [3]:
# Şema doğrulama: soru dosyası yazıldıktan sonra bu hücreyi çalıştırın.
from collections import Counter
from src.data import jsonl_oku

yol = cfg['veri']['sorular']  # data/processed/sorular.jsonl
sorular = jsonl_oku(yol)
gerekli = {'soru_id', 'soru', 'tip', 'kanit_chunk_id', 'varyant_grup', 'bolme'}
chunk_idler = {c.chunk_id for c in chunklar}

sorun = []
for s in sorular:
    eksik = gerekli - set(s)
    if eksik:
        sorun.append((s.get('soru_id'), f'eksik alan: {eksik}'))
    elif s['kanit_chunk_id'] not in chunk_idler:
        sorun.append((s['soru_id'], 'kanıt chunk bulunamadı'))

# varyant grubu bölme sızıntısı kontrolü
grup_bolme = {}
for s in sorular:
    grup_bolme.setdefault(s['varyant_grup'], set()).add(s['bolme'])
sizinti = [g for g, b in grup_bolme.items() if len(b) > 1]

print(f'{len(sorular)} soru denetlendi ({len(grup_bolme)} varyant grubu). Sorun: {sorun or "yok"}')
print(f'Bölme sızıntısı olan gruplar: {sizinti or "yok"}')
print('Bölme dağılımı:', dict(Counter(s['bolme'] for s in sorular)))
print('Soru tipi dağılımı:', dict(Counter(s['tip'] for s in sorular)))
assert not sorun and not sizinti, 'Önce veri sorunlarını düzeltin.'
print('Veri hazır. Sıradaki adım: 02_retrieval_deneyleri.ipynb')

328 soru denetlendi (82 varyant grubu). Sorun: yok
Bölme sızıntısı olan gruplar: yok
Bölme dağılımı: {'test': 120, 'train': 156, 'val': 52}
Soru tipi dağılımı: {'olgusal': 82, 'paraphrase': 82, 'bicimbilimsel': 82, 'cok-maddeli': 82}
Veri hazır. Sıradaki adım: 02_retrieval_deneyleri.ipynb
